In [3]:
# Loome SparkSession-i koos Kafka konnektori ja Delta Lake'i toega.
# spark.jars.packages laeb vajalikud JAR-id Maven Central-ist Sparki käivitumisel.
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("metalfab")
    .master("spark://6034.local:7077") \
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.11")
    .config("spark.driver.memory", "8g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}")

26/05/29 19:12:32 WARN SparkContext: Another SparkContext is being constructed (or threw an exception in its constructor). This may indicate an error, since only one SparkContext should be running in this JVM (see SPARK-2243). The other SparkContext was created at:
org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:59)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:75)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:53)
java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:502)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:486)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.command

Spark 4.1.2


In [ ]:
#df = spark.read.json("../data/lake/*/*/*/*/*/*")
    #.filter(F.col("tag")=="status").sort(F.col("timestamp"))

#df.show(100, truncate=False)

In [4]:
def write_to_postgres(batch_df, batch_id):
    db_url = "jdbc:postgresql://6034.local:5432/praktikum"
    db_properties = {
       "user": "praktikum",
       "password": "praktikum",
       "driver": "org.postgresql.Driver"
        }
    # Kirjutame selle konkreetse partii andmebaasi
    batch_df.write.jdbc(
       url=db_url,
       table="staging.raw_factory_data",
       mode="append",
       properties=db_properties
       )
    print(f"Batch {batch_id} kirjutatud andmebaasi.")

processed_df = spark.readStream \
    .schema("dept STRING, machine STRING, tag STRING, timestamp TIMESTAMP, topic STRING, value STRING") \
    .json("/Users/kerry.lumi/Documents/dev/smart-factory-analytics/data/lake/*/*/*/*/*/*")

# 4. Käivita striim
query = processed_df.writeStream \
.foreachBatch(write_to_postgres) \
.option("checkpointLocation", "./checkpoints/postgres_stream") \
.start()

#.option("maxFilesPerTrigger", 200) \
#query.awaitTermination()

26/05/29 19:12:54 WARN SharedInMemoryCache: Evicting cached table partition metadata from memory due to size constraints (spark.sql.hive.filesourcePartitionFileCacheSize = 262144000 bytes). This may impact query planning performance.
26/05/29 19:12:58 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/29 19:13:23 WARN FileStreamSource: Listed 1808998 file(s) in 19619 ms     
                                                                                

Batch 723 kirjutatud andmebaasi.


26/05/29 19:14:05 WARN FileStreamSource: Listed 1809882 file(s) in 22261 ms     
                                                                                

Batch 724 kirjutatud andmebaasi.


26/05/29 19:14:27 WARN FileStreamSource: Listed 1810397 file(s) in 19749 ms     
                                                                                

Batch 725 kirjutatud andmebaasi.


[Stage 11:=======================>                             (108 + 12) / 240]

In [9]:
query.status

{'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}

In [8]:
query.stop()

In [ ]:
"""
db_url = "jdbc:postgresql://db:5432/praktikum"
db_properties = {
   "user": "praktikum",
   "password": "praktikum",
   "driver": "org.postgresql.Driver"
    }
   
# 4. Kirjutamine andmebaasi
df.write.jdbc(
    url=db_url, 
    table="staging.raw_factory_data", 
    mode="overwrite",  # Valikud: append, overwrite, ignore, errorifexists
    properties=db_properties
   )
"""